In [53]:
from langchain.llms import OpenAIChat
from langchain import PromptTemplate, LLMChain
from langchain.embeddings import OpenAIEmbeddings
from langchain.callbacks import get_openai_callback
from langchain.vectorstores import Chroma
from langchain.document_loaders import TextLoader
from langchain.indexes import VectorstoreIndexCreator

In [46]:
import os
from os.path import abspath, join
import dotenv

In [52]:
dotenv.load_dotenv('../bin/.env')

True

In [49]:
dir = abspath('')

In [50]:
loader = TextLoader(join(dir, 'test_data', 's41540-017-0013-4.txt'))

In [51]:
# with open(join(dir, 'test_data', 's41540-017-0013-4.txt'), 'r') as f:
#     full_text = f.read()

# find the abstract of an article

In [32]:
query = """The following text was extracted from a PDF document.

Does this portion include part of the article's Abstract?"""

In [36]:
query_embeddings = embeddings.embed_query(query)

used 0 tokens, cost $0.000


In [35]:
query_embeddings

[-0.010806171223521233,
 0.0029074379708617926,
 0.007139910478144884,
 -0.03273067995905876,
 -0.019779212772846222,
 0.03347455710172653,
 -0.012466615997254848,
 -0.013615643605589867,
 -0.03185396268963814,
 -0.012991316616535187,
 0.018530558794736862,
 0.036078132688999176,
 -0.01305773388594389,
 0.014346239157021046,
 -0.003935253247618675,
 0.020549658685922623,
 0.02251562476158142,
 -0.008142818696796894,
 0.007910356856882572,
 -0.014452507719397545,
 -0.0018381119007244706,
 0.01117146946489811,
 -0.004941482096910477,
 0.006724799517542124,
 -0.0021436335518956184,
 0.01207475084811449,
 0.023671293631196022,
 -0.030791278928518295,
 0.004566221963614225,
 -0.002929023699834943,
 -0.0041544316336512566,
 0.0031365794129669666,
 -0.027496958151459694,
 -0.005074318032711744,
 0.0009124141652137041,
 0.0007173119229264557,
 -0.01004236750304699,
 -0.03942558914422989,
 0.028214270249009132,
 -0.019858913496136665,
 0.021638911217451096,
 0.010035725310444832,
 -0.0011540087

In [54]:
index = VectorstoreIndexCreator().from_loaders([loader])

ValueError: Could not import chromadb python package. Please install it with `pip install chromadb`.

In [39]:
db = Chroma.from_documents(texts, document_embeddings)

56

# categorize

In [23]:
options = ['chemicals', 'biochemical reactions', 'species',
           'countries', 'geological ages', 'ignition fuel',
           'planetary objects',
           'NASA projects',
           'proteins',
           'diseases', # https://disease-ontology.org
           'pharmaceutical drugs', # https://bioportal.bioontology.org/ontologies/DRON
          ]

In [24]:
template = """The following text was extracted from a PDF document:

{text}

We are trying to categorize the document according to categories it
will discuss. The category options are:

{options}

Based on the text extract, list the categories that are likely to
exist in the full text of the article. List one category on each line.
Only provide categories from the category options list.
If no categories are expected, say 'None found'.
"""

In [25]:
# template = """The following text was extracted from a PDF document:

# {text}

# We are trying to categorize the document according to categories it
# will discuss. The category options are:

# {options}

# Based on the text extract, list the categories that are likely to
# exist in the full text of the article. List one category on each line.
# Include any additional categories you can think of that seem relevant.
# If no categories are expected, say 'None found'.
# """

In [26]:
llm = OpenAIChat(temperature=0)

In [27]:
prompt = PromptTemplate(template=template, input_variables=["text", "options"])
llm_chain = LLMChain(prompt=prompt, llm=llm)
with get_openai_callback() as cb:
    # TODO how to get the right first text blob? should use embeddings
    print(llm_chain.run(text=full_text[0:3000], options=', '.join(options)))
    print(f'used {cb.total_tokens} tokens, cost ${cb.total_tokens/1000 * 0.002:.3f}')



biochemical reactions
species
pharmaceutical drugs
used 810 tokens, cost $0.002


# find stuff

In [8]:
template = """The following text was extracted from a PDF document:

{text}

List the chemical compounds that are mentioned in the article. If you
find a chemical, provide the name of each chemical on a new line with
a description, like (chemical Name: Description). If you do not find a
chemical, say "No chemicals found".
"""
prompt = PromptTemplate(template=template, input_variables=["text"])
llm_chain = LLMChain(prompt=prompt, llm=llm)
for i in range(2):
    with get_openai_callback() as cb:
        print(llm_chain.run(full_text[2000*(i):2000*(i+1)]).replace("No chemicals found.", "").strip())
        print(cb)
        print(f'used {cb.total_tokens} tokens, cost ${cb.total_tokens/1000 * 0.002:.3f}')

vinblastine: an anti-cancer therapeutic
vincristine: an anti-cancer therapeutic
Monoterpene indole alkaloids (MIAs): a diverse family of complex plant secondary metabolites
vindoline: a precursor to vinblastine and catharanthine
catharanthine: a precursor to vinblastine and vindoline
geranyl pyrophosphate: a yeast native metabolite
tryptophan: a yeast native metabolite
used 738 tokens, cost $0.001

used 514 tokens, cost $0.001


In [42]:
template = """The following text was extracted from a PDF document:

{text}

List the organisms that are being studied in the article. If you
find an organism, provide the common name and scientific name of
each organism on a new line with, like:

- Common Name (Scientific Name).

If you do not find an organism, say "No organisms found".
"""
prompt = PromptTemplate(template=template, input_variables=["text"])
llm_chain = LLMChain(prompt=prompt, llm=llm)
for t in map(''.join, zip(*[iter(full_text)]*2000)):
    print(llm_chain.run(t).replace("No organisms found.", "").replace('-', '').strip())

Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
Yeast (Saccharomyces cerevisiae)
 Plants (Gentianales plants, Catharanthus roseus)
Baker's yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
Yeast (Saccharomyces cerevisiae)

Madagascar periwinkle (Catharanthus roseus)
 Yeast (Saccharomyces cerevisiae)



Yeast (Saccharomyces cerevisiae)
 Rauvolfia tetraphylla
 Sesamum indicum
 Catharanthus roseus
 Populus trichocarpa
Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
 Indian snakeroot (Rauvolfia serpentina)
Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)

Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)


Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
Mexican Tab

In [22]:
text = f"""The following text was extracted from a PDF document:

{full_text[1000:2000]}

List the model organisms being studies that are mentioned in the article. If you
find organisms, provide each organism scientific name on a new line with no other
information. If you do not find a model organism, say "No organisms found".
"""
print(llm(text))


Saccharomyces cerevisiae
Catharanthus roseus


In [30]:
full_text.index('SpyCas9')

29081

In [37]:
text = f"""The following text was extracted from a PDF document:

{full_text[29000:31000]}

List the engineered strains that are mentioned in the article. If you
find an engineered strain, provide the name of each strain on a new line with
a description, like (Strain Name: Description). If you do not find a
strain, say "No strains found".
"""
print(llm(text))


MIA-CM-3: De novo strictosidine production strain
MIA-CR-A: Strain producing 11.5 μg l−1 tabersonine
MIA-CW-1: Strain producing 0.137 μg l−1 vindoline
MIA-EM-1: Biphasic fermentation strain producing 2.32 μg l−1 vindoline and 2.82 μg l−1 catharanthine


In [15]:
paper_path = join(dir, '..', 'data', 's41586-022-05157-3.pdf')